# FL Shapley 20 Newsgroups – Quick Analysis Notebook

This notebook provides interactive exploration of the experiment outputs.
Run `run_experiments.py` first to generate the CSV and PNG files.

**Structure**
1. Load outputs
2. Accuracy curves comparison
3. Shapley heatmaps (per condition)
4. Top contributing classes
5. Suspicious client detection
6. Cross-condition Shapley shift

In [ ]:
import os, sys
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
matplotlib.rcParams['figure.dpi'] = 120

OUTPUT_BASE = '../outputs'
ATTACKS = ['clean', 'freerider', 'poisoning']

def load_csv(attack, filename):
    path = os.path.join(OUTPUT_BASE, attack, filename)
    if os.path.exists(path):
        return pd.read_csv(path)
    print(f'Not found: {path}')
    return None

# Load all outputs
metrics   = {a: load_csv(a, 'round_metrics.csv')           for a in ATTACKS}
shapley   = {a: load_csv(a, 'class_shapley_by_round.csv')  for a in ATTACKS}
dist_df   = pd.read_csv(os.path.join(OUTPUT_BASE, 'client_data_distribution.csv'))

print('Loaded!')
print('Distribution shape:', dist_df.shape)

## 1. Client Data Distribution

In [ ]:
pivot = dist_df.pivot_table(index='client_id', columns='class_name', values='sample_count', fill_value=0)
fig, ax = plt.subplots(figsize=(16, 5))
im = ax.imshow(pivot.values, aspect='auto', cmap='YlOrRd')
ax.set_xticks(range(len(pivot.columns)))
ax.set_xticklabels(pivot.columns, rotation=45, ha='right', fontsize=8)
ax.set_yticks(range(len(pivot.index)))
ax.set_yticklabels([f'Client {i}' for i in pivot.index], fontsize=9)
ax.set_title('Client Data Distribution (Dirichlet non-IID)', fontsize=13)
plt.colorbar(im, ax=ax, label='Sample Count')
plt.tight_layout()
plt.show()

## 2. Accuracy vs Round (All Conditions)

In [ ]:
COLOURS = {'clean': '#2196F3', 'freerider': '#FF9800', 'poisoning': '#F44336'}

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
for attack, df in metrics.items():
    if df is None: continue
    c = COLOURS[attack]
    axes[0].plot(df['round'], df['global_accuracy'], label=attack, color=c, marker='o', ms=3)
    axes[1].plot(df['round'], df['global_macro_f1'], label=attack, color=c, marker='s', ms=3)

for ax, title in zip(axes, ['Validation Accuracy', 'Macro-F1']):
    ax.set_xlabel('Round'); ax.set_ylabel(title)
    ax.legend(); ax.grid(alpha=0.3); ax.set_ylim(0,1)

plt.tight_layout(); plt.show()

## 3. Shapley Heatmap — Clean

In [ ]:
def shapley_heatmap(df, title):
    if df is None: return
    agg = df.groupby(['round','class_name'])['shapley_value'].mean().reset_index()
    pivot = agg.pivot(index='class_name', columns='round', values='shapley_value')
    pivot = pivot.loc[pivot.mean(axis=1).sort_values(ascending=False).index]
    vmax = max(abs(pivot.values[~np.isnan(pivot.values)]).max(), 1e-6)
    fig, ax = plt.subplots(figsize=(14, 7))
    im = ax.imshow(pivot.fillna(0).values, aspect='auto', cmap='RdBu_r', vmin=-vmax, vmax=vmax)
    ax.set_xticks(range(pivot.shape[1]))
    ax.set_xticklabels(pivot.columns, rotation=45, fontsize=7)
    ax.set_yticks(range(pivot.shape[0]))
    ax.set_yticklabels(pivot.index, fontsize=8)
    ax.set_xlabel('Round'); ax.set_ylabel('Class')
    ax.set_title(title, fontsize=13)
    plt.colorbar(im, ax=ax, label='Mean Shapley')
    plt.tight_layout(); plt.show()

shapley_heatmap(shapley['clean'], 'Shapley Heatmap – Clean')

## 4. Shapley Heatmap — Free-rider

In [ ]:
shapley_heatmap(shapley['freerider'], 'Shapley Heatmap – Free-rider Attack')

## 5. Shapley Heatmap — Poisoning

In [ ]:
shapley_heatmap(shapley['poisoning'], 'Shapley Heatmap – Poisoning Attack')

## 6. Top Contributing Classes

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for ax, (attack, df) in zip(axes, shapley.items()):
    if df is None: ax.set_visible(False); continue
    mean_sv = df.groupby('class_name')['shapley_value'].mean().sort_values()
    colours = ['#F44336' if v < 0 else '#2196F3' for v in mean_sv.values]
    ax.barh(mean_sv.index, mean_sv.values, color=colours)
    ax.axvline(0, color='k', lw=0.8)
    ax.set_title(f'Mean Shapley – {attack}', fontsize=11)
    ax.set_xlabel('Mean Shapley Value')
    ax.tick_params(axis='y', labelsize=7)
    ax.grid(axis='x', alpha=0.3)

plt.tight_layout(); plt.show()

## 7. Suspicious Client Detection

In [ ]:
THRESHOLD = 0.003
for attack, df in shapley.items():
    if df is None: continue
    stats = df.groupby('client_id')['shapley_value'].agg(
        mean_abs=lambda x: abs(x).mean(), mean='mean'
    )
    suspicious = stats[stats['mean_abs'] < THRESHOLD]
    print(f'\n=== {attack.upper()} ===')
    if suspicious.empty:
        print(f'  No suspicious clients (threshold={THRESHOLD})')
    else:
        print(suspicious.round(6).to_string())

## 8. Per-client Contribution Trends

In [ ]:
attack = 'freerider'
df = shapley.get(attack)
if df is not None:
    cr = df.groupby(['round','client_id'])['shapley_value'].sum().reset_index()
    fig, ax = plt.subplots(figsize=(11, 4))
    for cid in sorted(cr['client_id'].unique()):
        sub = cr[cr['client_id'] == cid]
        ax.plot(sub['round'], sub['shapley_value'], label=f'C{cid}', alpha=0.7)
    ax.axhline(0, color='k', lw=0.8, ls=':')
    ax.set_xlabel('Round'); ax.set_ylabel('Total Shapley (client)')
    ax.set_title(f'Per-Client Contribution – {attack}'); ax.legend(fontsize=7, ncol=5)
    ax.grid(alpha=0.3)
    plt.tight_layout(); plt.show()